# BBC News: Business Category Analysis
## Traditional NLP Pipeline: TF-IDF + LDA Sub-category Discovery


In [1]:
# Core libraries
import os
import re
import sys

# Data handling
import pandas as pd
import numpy as np

# NLP — Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# NLP — Topic Modelling
from sklearn.decomposition import LatentDirichletAllocation

# Display
from rich.console import Console
from rich.table import Table

# Add parent directory to path so we can access loader.py
sys.path.append('..')
from loader import load_data

## Load the Data for Business only

In [2]:
# Load all articles and labels
documents, labels = load_data('../data')

# Filter business articles only
business_docs = [documents[i] for i in range(len(documents)) if labels[i] == 'business']

print(f"Total articles in dataset: {len(documents)}")
print(f"Business articles: {len(business_docs)}")
print(f"\nSample business article:\n{business_docs[0][:300]}")

Total articles in dataset: 2225
Business articles: 510

Sample business article:
UK economy facing 'major risks'

The UK manufacturing sector will continue to face "serious challenges" over the next two years, the British Chamber of Commerce (BCC) has said.

The group's quarterly survey of companies found exports had picked up in the last three months of 2004 to their best level


## Clean the Data


In [3]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def clean_text(text):
    # Lowercase everything
    text = text.lower()
    
    # Remove text inside square brackets
    text = re.sub(r'\[.*?\]', '', text)
    
    # Remove non-letter characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    
    return ' '.join(tokens)

# Apply cleaning to business articles
cleaned_business = [clean_text(doc) for doc in business_docs]

print(f"Cleaning complete — {len(cleaned_business)} business articles cleaned")
print(f"\nBefore cleaning:\n{business_docs[0][:200]}")
print(f"\nAfter cleaning:\n{cleaned_business[0][:200]}")

Cleaning complete — 510 business articles cleaned

Before cleaning:
UK economy facing 'major risks'

The UK manufacturing sector will continue to face "serious challenges" over the next two years, the British Chamber of Commerce (BCC) has said.

The group's quarterly 

After cleaning:
uk economy facing major risks uk manufacturing sector continue face challenges years british chamber commerce bcc said groups quarterly survey companies exports picked months best levels years rise ca


## Remove Duplicates

In [4]:
# Remove duplicates from business articles
seen = set()
unique_business = []

for doc in cleaned_business:
    if doc not in seen:
        seen.add(doc)
        unique_business.append(doc)

print(f"Before deduplication: {len(cleaned_business)} articles")
print(f"After deduplication: {len(unique_business)} articles")
print(f"Duplicates removed: {len(cleaned_business) - len(unique_business)}")

Before deduplication: 510 articles
After deduplication: 503 articles
Duplicates removed: 7
